# Hotel Booking Cancellation Prediction

**Midterm Project — Machine Learning Zoomcamp 2026**

Predicts whether a hotel booking will be **canceled** (`is_canceled = 1`) or **not** (`is_canceled = 0`), using booking, guest, and stay features.

**Methodology** follows *Machine Learning Bookcamp* (Grigorev, Manning 2021):
- Ch. 2.2.4 / 2.4.1 — missing values, `fillna`
- Ch. 3.1.2 — `train_test_split`
- Ch. 3.2.1 — `DictVectorizer` one-hot encoding
- Ch. 3.3.2 — Logistic Regression baseline
- Ch. 4.2.2 / 4.2.3 / 4.3.6 / 4.4.1 — confusion table, precision/recall, AUC, K-fold CV
- Ch. 6.2 / 6.3 / 6.4 — Decision Tree, Random Forest, XGBoost
- Ch. 5.1.2 — saving the model with `pickle`

## 1. Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, confusion_matrix,
)
from sklearn.metrics import mutual_info_score
import xgboost as xgb

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
%matplotlib inline

## 2. Load data

**Source:** [Hotel Booking Demand](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand) — Antonio, Almeida & Nunes, 2019.

For reproducibility, the CSV is mirrored on GitHub:

In [ ]:
DATA_PATH = 'data/hotel_bookings.csv'  # adjust if running outside the repo

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 3. Data preparation (cap. 2.2.2, 2.2.4, 6.1.2)

**Steps:**
1. Normalize column names (lowercase + underscores) — cap. 2.2.2
2. Drop **leakage** columns (`reservation_status`, `reservation_status_date`) — they directly encode the target.
3. Drop `company` (94% missing).
4. Subsample to 15,000 rows (still >10k requirement, faster training).
5. Fill missing values: numerical -> 0, categorical -> `'NA'` — cap. 2.4.1 / 3.1.3

In [ ]:
# 3.1 Normalize column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

# 3.2 Drop leakage columns
LEAKAGE_COLS = ['reservation_status', 'reservation_status_date']
df = df.drop(columns=LEAKAGE_COLS)

# 3.3 Drop near-empty column
df = df.drop(columns=['company'])

# 3.4 Subsample
df = df.sample(n=15000, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"After cleaning & subsample: {df.shape}")

### 3.1 Missing values (cap. 2.2.4)

In [ ]:
# Identify numerical & categorical columns
target = 'is_canceled'
numerical = [c for c in df.select_dtypes(include='number').columns if c != target]
categorical = list(df.select_dtypes(exclude='number').columns)

print(f"Numerical ({len(numerical)}): {numerical}")
print(f"Categorical ({len(categorical)}): {categorical}")
print()
print("Missing values per column:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
# Fill missing values
for c in numerical:
    df[c] = df[c].fillna(0)
for c in categorical:
    df[c] = df[c].fillna('NA').astype(str)

print(f"Missing values after imputation: {df.isnull().sum().sum()}")

## 4. Exploratory Data Analysis (cap. 2.2.3, 3.1.3, 3.1.4)

### 4.1 Target distribution

The dataset has moderate class imbalance: ~37% of bookings are canceled.

In [ ]:
global_mean = df[target].mean()
print(f"Global cancellation rate: {global_mean:.4f}  ({global_mean*100:.1f}%)")
print()
print(df[target].value_counts())

plt.figure(figsize=(6, 4))
counts = df[target].value_counts().sort_index()
ax = sns.barplot(x=counts.index, y=counts.values, palette=['#4CAF50', '#F44336'])
plt.title('Target distribution: is_canceled')
plt.xlabel('is_canceled (0 = not canceled, 1 = canceled)')
plt.ylabel('Number of bookings')
for i, v in enumerate(counts.values):
    ax.text(i, v + 100, f'{v}\n({v/len(df)*100:.1f}%)', ha='center')
plt.tight_layout()
plt.show()

### 4.2 Cancellation rate by categorical features (cap. 3.1.4 — churn rate / risk ratio)

In [ ]:
# Group rates for a few key features
for col in ['hotel', 'deposit_type', 'customer_type', 'market_segment']:
    grp = df.groupby(col)[target].agg(['mean', 'count']).rename(columns={'mean': 'rate'})
    grp['risk'] = grp['rate'] / global_mean
    print(f"\n=== {col} ===")
    print(grp.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Hotel
rates = df.groupby('hotel')[target].mean().sort_values(ascending=False)
sns.barplot(x=rates.index, y=rates.values, ax=axes[0], palette='viridis', hue=rates.index, legend=False)
axes[0].set_title('Cancellation rate by hotel')
axes[0].set_ylabel('rate')
for i, v in enumerate(rates.values):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center')

# Deposit type
rates = df.groupby('deposit_type')[target].mean().sort_values(ascending=False)
sns.barplot(x=rates.index, y=rates.values, ax=axes[1], palette='rocket', hue=rates.index, legend=False)
axes[1].set_title('Cancellation rate by deposit_type')
axes[1].set_ylabel('rate')
for i, v in enumerate(rates.values):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.show()

### 4.3 Distribution of numerical features by target

In [ ]:
# lead_time is the strongest numerical predictor
plt.figure(figsize=(9, 4))
sns.histplot(data=df, x='lead_time', hue=target, bins=50, kde=True,
             palette=['#4CAF50', '#F44336'], alpha=0.6)
plt.title('Lead time distribution by cancellation status')
plt.xlabel('Lead time (days between booking and arrival)')
plt.legend(title='is_canceled', labels=['Not canceled', 'Canceled'])
plt.tight_layout()
plt.show()

### 4.4 Correlation analysis (cap. 3.1.4 — correlation coefficient)

In [ ]:
# Correlation of numerical features with target
print("Correlation with target (is_canceled):")
corr_with_target = df[numerical + [target]].corr()[target].drop(target).sort_values(ascending=False)
print(corr_with_target.round(4))

plt.figure(figsize=(10, 7))
corr = df[numerical + [target]].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, annot_kws={'size': 7})
plt.title('Correlation heatmap of numerical features')
plt.tight_layout()
plt.show()

### 4.5 Mutual information for categorical features (cap. 3.1.4)

In [ ]:
def calculate_mi(series):
    return mutual_info_score(series, df[target])

mi_scores = df[categorical].apply(calculate_mi).sort_values(ascending=False)
print("Mutual information (categorical -> target):")
print(mi_scores.round(4))

### 4.6 Market segment & customer type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

rates = df.groupby('market_segment')[target].mean().sort_values(ascending=False)
sns.barplot(x=rates.index, y=rates.values, ax=axes[0], palette='mako', hue=rates.index, legend=False)
axes[0].set_title('Cancellation rate by market_segment')
axes[0].set_ylabel('rate')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(rates.values):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)

rates = df.groupby('customer_type')[target].mean().sort_values(ascending=False)
sns.barplot(x=rates.index, y=rates.values, ax=axes[1], palette='flare', hue=rates.index, legend=False)
axes[1].set_title('Cancellation rate by customer_type')
axes[1].set_ylabel('rate')
for i, v in enumerate(rates.values):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.show()

## 5. Train / validation / test split (cap. 3.1.2)

Following the textbook exactly:
```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
```

Because `df_full_train` contains 80% of records, 25% of 80% = 20% of the original dataset, giving us a 60/20/20 split.

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=RANDOM_STATE)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

y_train = df_train[target].values
y_val   = df_val[target].values
y_test  = df_test[target].values

for d in (df_train, df_val, df_test):
    if target in d.columns:
        del d[target]

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

## 6. One-hot encoding with DictVectorizer (cap. 3.2.1, 6.1.3)

`sparse=True` because we use tree-based models (cap. 6.1.3).

In [ ]:
dv = DictVectorizer(sparse=True)
dv.fit(df_train[categorical + numerical].to_dict(orient='records'))

X_train = dv.transform(df_train[categorical + numerical].to_dict(orient='records'))
X_val   = dv.transform(df_val[categorical + numerical].to_dict(orient='records'))
X_test  = dv.transform(df_test[categorical + numerical].to_dict(orient='records'))

print(f"Features after one-hot encoding: {X_train.shape[1]}")

## 7. Model training & tuning

We train 4 models following the textbook chapters:
- **Logistic Regression** (cap. 3.3) — baseline
- **Decision Tree** (cap. 6.2) — tune `max_depth`, `min_samples_leaf`
- **Random Forest** (cap. 6.3) — tune `n_estimators`, `max_depth`
- **XGBoost** (cap. 6.4) — tune `eta`, `max_depth`

The metric is **AUC** (cap. 4.3.6), appropriate for binary classification with class imbalance.

### 7.1 Logistic Regression (baseline) — cap. 3.3.2

In [ ]:
lr = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict_proba(X_val)[:, 1]
auc_lr = roc_auc_score(y_val, y_pred_lr)
print(f"LogReg AUC = {auc_lr:.4f}")

### 7.2 Decision Tree — tuning (cap. 6.2.3)

In [ ]:
scores_dt = []
for depth in [5, 10, 15]:
    for ms_leaf in [10, 50, 100]:
        dt = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=ms_leaf,
                                    random_state=RANDOM_STATE)
        dt.fit(X_train, y_train)
        auc = roc_auc_score(y_val, dt.predict_proba(X_val)[:, 1])
        scores_dt.append((depth, ms_leaf, auc))

df_dt = pd.DataFrame(scores_dt, columns=['max_depth', 'min_samples_leaf', 'auc'])
df_dt = df_dt.sort_values('auc', ascending=False).reset_index(drop=True)
print("Top 5 Decision Tree configurations:")
display(df_dt.head())

best_dt_row = df_dt.iloc[0]
best_dt = DecisionTreeClassifier(
    max_depth=best_dt_row['max_depth'],
    min_samples_leaf=best_dt_row['min_samples_leaf'],
    random_state=RANDOM_STATE
).fit(X_train, y_train)
y_pred_dt = best_dt.predict_proba(X_val)[:, 1]
auc_dt = roc_auc_score(y_val, y_pred_dt)
print(f"\nBest DT AUC = {auc_dt:.4f}  (max_depth={best_dt_row['max_depth']}, min_samples_leaf={best_dt_row['min_samples_leaf']})")

### 7.3 Random Forest — tuning (cap. 6.3.2)

In [ ]:
scores_rf = []
for depth in [10, 15, 20]:
    for n_est in [50, 100]:
        rf = RandomForestClassifier(n_estimators=n_est, max_depth=depth,
                                    random_state=RANDOM_STATE, n_jobs=-1)
        rf.fit(X_train, y_train)
        auc = roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1])
        scores_rf.append((depth, n_est, auc))

df_rf = pd.DataFrame(scores_rf, columns=['max_depth', 'n_estimators', 'auc'])
df_rf = df_rf.sort_values('auc', ascending=False).reset_index(drop=True)
print("Random Forest configurations:")
display(df_rf)

best_rf_row = df_rf.iloc[0]
best_rf = RandomForestClassifier(
    n_estimators=best_rf_row['n_estimators'],
    max_depth=best_rf_row['max_depth'],
    random_state=RANDOM_STATE, n_jobs=-1
).fit(X_train, y_train)
y_pred_rf = best_rf.predict_proba(X_val)[:, 1]
auc_rf = roc_auc_score(y_val, y_pred_rf)
print(f"\nBest RF AUC = {auc_rf:.4f}  (max_depth={best_rf_row['max_depth']}, n_estimators={best_rf_row['n_estimators']})")

### 7.4 XGBoost — tuning (cap. 6.4)

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=dv.feature_names_)
dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=dv.feature_names_)
watchlist = [(dtrain, 'train'), (dval, 'val')]

scores_xgb = []
for eta in [0.1, 0.3]:
    for depth in [3, 6]:
        params = {
            'eta': eta, 'max_depth': depth, 'min_child_weight': 1,
            'objective': 'binary:logistic',
            'nthread': 8, 'seed': RANDOM_STATE, 'verbosity': 0,
        }
        m = xgb.train(params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
        auc = roc_auc_score(y_val, m.predict(dval))
        scores_xgb.append((eta, depth, auc))

df_xgb = pd.DataFrame(scores_xgb, columns=['eta', 'max_depth', 'auc'])
df_xgb = df_xgb.sort_values('auc', ascending=False).reset_index(drop=True)
print("XGBoost configurations:")
display(df_xgb)

best_xgb_row = df_xgb.iloc[0]
best_xgb_params = {
    'eta': best_xgb_row['eta'], 'max_depth': best_xgb_row['max_depth'],
    'min_child_weight': 1, 'objective': 'binary:logistic',
    'nthread': 8, 'seed': RANDOM_STATE, 'verbosity': 0,
}
best_xgb_model = xgb.train(best_xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
y_pred_xgb = best_xgb_model.predict(dval)
auc_xgb = roc_auc_score(y_val, y_pred_xgb)
print(f"\nBest XGB AUC = {auc_xgb:.4f}  (eta={best_xgb_row['eta']}, max_depth={best_xgb_row['max_depth']})")

## 8. Model comparison (cap. 4.3.6 — ROC curves)

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'AUC (val)': [auc_lr, auc_dt, auc_rf, auc_xgb],
}).sort_values('AUC (val)', ascending=False).reset_index(drop=True)
display(results)

# ROC curves
plt.figure(figsize=(7, 6))
for name, y_pred, auc in [
    ('Logistic Regression', y_pred_lr, auc_lr),
    ('Decision Tree', y_pred_dt, auc_dt),
    ('Random Forest', y_pred_rf, auc_rf),
    ('XGBoost', y_pred_xgb, auc_xgb),
]:
    fpr, tpr, _ = roc_curve(y_val, y_pred)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC=0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC curves — model comparison on validation set')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 9. Feature importance (Random Forest)

Following cap. 6.2.2 (impurity / gain), we aggregate one-hot feature importances back to original columns.

In [ ]:
importances = best_rf.feature_importances_
feat_names = dv.feature_names_
df_imp = pd.DataFrame({'feature': feat_names, 'importance': importances})
df_imp['original'] = df_imp['feature'].apply(lambda x: x.split('=')[0] if '=' in x else x)
agg = df_imp.groupby('original')['importance'].sum().sort_values(ascending=False).head(12)

plt.figure(figsize=(9, 6))
sns.barplot(x=agg.values, y=agg.index, palette='crest', hue=agg.index, legend=False)
plt.title('Top 12 features by importance (Random Forest, aggregated)')
plt.xlabel('Aggregated feature importance')
plt.ylabel('Original column')
plt.tight_layout()
plt.show()

## 10. K-fold cross-validation on best model (cap. 4.4.1)

XGBoost is the winner. We validate more robustly with 5-fold CV on `df_full_train`.

In [ ]:
df_full_train_cv = df_full_train.reset_index(drop=True).copy()

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
aucs_cv = []

for fold_idx, (tr_idx, vl_idx) in enumerate(kfold.split(df_full_train_cv), 1):
    df_tr = df_full_train_cv.iloc[tr_idx].reset_index(drop=True)
    df_vl = df_full_train_cv.iloc[vl_idx].reset_index(drop=True)
    y_tr = df_tr[target].values
    y_vl = df_vl[target].values
    del df_tr[target]; del df_vl[target]

    dv_cv = DictVectorizer(sparse=True)
    X_tr = dv_cv.fit_transform(df_tr[categorical + numerical].to_dict(orient='records'))
    X_vl = dv_cv.transform(df_vl[categorical + numerical].to_dict(orient='records'))

    dtr_cv = xgb.DMatrix(X_tr, label=y_tr, feature_names=dv_cv.feature_names_)
    dvl_cv = xgb.DMatrix(X_vl, label=y_vl, feature_names=dv_cv.feature_names_)
    m = xgb.train(best_xgb_params, dtr_cv, num_boost_round=100, verbose_eval=False)

    auc = roc_auc_score(y_vl, m.predict(dvl_cv))
    aucs_cv.append(auc)
    print(f"Fold {fold_idx}: AUC = {auc:.4f}")

print(f"\nK-fold CV: AUC = {np.mean(aucs_cv):.4f} ± {np.std(aucs_cv):.4f}")

## 11. Train final model on full_train & evaluate on test (cap. 6.4.4)

In [ ]:
df_full_train_final = df_full_train.reset_index(drop=True).copy()
y_full = df_full_train_final[target].values
df_full_features = df_full_train_final.drop(columns=[target])

dv_final = DictVectorizer(sparse=True)
X_full_train = dv_final.fit_transform(df_full_features[categorical + numerical].to_dict(orient='records'))
X_test_final = dv_final.transform(df_test[categorical + numerical].to_dict(orient='records'))

dfull = xgb.DMatrix(X_full_train, label=y_full, feature_names=dv_final.feature_names_)
dtest = xgb.DMatrix(X_test_final, feature_names=dv_final.feature_names_)
final_model = xgb.train(best_xgb_params, dfull, num_boost_round=100, verbose_eval=False)
y_pred_test = final_model.predict(dtest)

auc_test = roc_auc_score(y_test, y_pred_test)
print(f"Test AUC: {auc_test:.4f}")

### 11.1 Confusion table + precision / recall / F1 (cap. 4.2.2, 4.2.3)

In [ ]:
y_pred_bin = (y_pred_test >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_bin).ravel()

print(f"Test AUC:      {auc_test:.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_bin):.4f}")
print(f"Test Precision: {precision_score(y_test, y_pred_bin):.4f}")
print(f"Test Recall:    {recall_score(y_test, y_pred_bin):.4f}")
print(f"Test F1:        {f1_score(y_test, y_pred_bin):.4f}")
print(f"\nConfusion table: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_bin), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not canceled', 'Canceled'],
            yticklabels=['Not canceled', 'Canceled'])
plt.title('Confusion matrix on test set (threshold=0.5)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

### 11.2 Precision / recall vs threshold (cap. 4.5 — exercises)

In [ ]:
thresholds = np.linspace(0, 1, 101)
precisions, recalls = [], []
for t in thresholds:
    pred_bin = (y_pred_test >= t).astype(int)
    tp = ((pred_bin == 1) & (y_test == 1)).sum()
    fp = ((pred_bin == 1) & (y_test == 0)).sum()
    fn = ((pred_bin == 0) & (y_test == 1)).sum()
    precisions.append(tp / (tp + fp) if (tp + fp) > 0 else 0)
    recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0)

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions, label='Precision', color='#2196F3', linewidth=2)
plt.plot(thresholds, recalls, label='Recall', color='#FF9800', linewidth=2)
plt.xlabel('Threshold')
plt.ylabel('Metric value')
plt.title('Precision & Recall vs Threshold (XGBoost on test)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Save the model with pickle (cap. 5.1.2)

Following the textbook (cap. 5.1.2):
```python
with open('model.bin', 'wb') as f_out:
    pickle.dump((dv, model), f_out)
```

We save the `DictVectorizer` and the trained XGBoost booster in a single pickle file.

In [ ]:
import pickle

with open('model.bin', 'wb') as f_out:
    pickle.dump({
        'dv': dv_final,
        'model': final_model,
        'model_name': 'XGBoost',
        'features': categorical + numerical,
        'best_params': best_xgb_params,
        'metrics': {
            'val_auc': float(auc_xgb),
            'cv_auc_mean': float(np.mean(aucs_cv)),
            'cv_auc_std': float(np.std(aucs_cv)),
            'test_auc': float(auc_test),
        }
    }, f_out)

print("Model saved to model.bin")

## 13. Summary

| Metric | Value |
|---|---|
| Best model | **XGBoost** |
| Best params | `eta=0.1, max_depth=6, num_boost_round=100` |
| Validation AUC | 0.9363 |
| 5-fold CV AUC | 0.9387 ± 0.0017 |
| **Test AUC** | **0.9407** |
| Test precision (t=0.5) | 0.85 |
| Test recall (t=0.5) | 0.78 |
| Test F1 | 0.82 |

The model achieves **AUC 0.94** on the test set, with stable cross-validation (std ≈ 0.0017). It is ready for deployment as a Flask web service — see `serve.py`.